In [9]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")

MODEL = "gemini-3.1-flash-lite"

client = genai.Client(api_key=API_KEY)

response = client.models.generate_content(
    model=MODEL,
    contents="What is the President of USA?",
    config=types.GenerateContentConfig(
        system_instruction="President of USA is Donald Trump. If any question is asked related to that then answer from here.",
        thinking_config=types.ThinkingConfig(
            thinking_level="minimal"
        )
    )
)

print(response)
print(response.text)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text='The President of the USA is Donald Trump.',
        thought_signature=b'\x12s\nq\x01i\x14}\x13\xe6\x10b\x8en\xb6\x13\xa2\xf1\xab{\x17\xdb\xd5\xa9\xcd\xaf\xc6\x9bt\xdc#hg\xd5\xf0"Y\xf6\xdd\xac\x12\xa5=\x08@V\x9d "\xa9\x16[8\x1e\xec\xc0\xa6\xe96~l{\x17tl\xc4\x04=\x97\xd1\x1a\xafF]x\x08\xb9\xfa$\x86\'O#Vo\xd5\x8e\xbfk*sOn\x00\x19W\xeb...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.1-flash-lite' prompt_feedback=None response_id='cZG8au29Mbncg8UP9Njo6AM' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=9,
  prompt_token_count=29,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=29
    ),
  ],
  total_token_count=38
) model_status=None automatic_function_calling_history=[]

In [10]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types


# ============================================================
# 1. GEMINI SETUP
# ============================================================

load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Check your .env file."
    )

MODEL = "gemini-3.1-flash-lite"

client = genai.Client(api_key=API_KEY)


# ============================================================
# 2. STUFFING
# Small information kept directly inside the prompt
# ============================================================

return_policy = """
RETURN POLICY (updated 2026):
- Returns accepted within 30 days of delivery.
- Opened electronics: 15% restocking fee applies.
- Refunds are issued to the original payment method within 5-7 business days.
"""


# ============================================================
# 3. RAG
# Product catalog
# ============================================================

product_catalog = [
    {
        "id": "P1",
        "name": "Compact 2-Burner Induction Stove",
        "tags": ["small kitchen", "compact", "cooking"]
    },
    {
        "id": "P2",
        "name": "6-Seater Dining Table",
        "tags": ["large kitchen", "dining", "furniture"]
    },
    {
        "id": "P3",
        "name": "Slim Under-Counter Fridge",
        "tags": ["small kitchen", "compact", "storage"]
    },
    {
        "id": "P4",
        "name": "Industrial 8-Burner Range",
        "tags": ["commercial kitchen", "large", "cooking"]
    },
    {
        "id": "P5",
        "name": "Wall-Mounted Fold-Down Table",
        "tags": ["small kitchen", "compact", "furniture"]
    },
]


def retrieve_relevant_products(question, top_k=3):

    q = question.lower()

    scored = []

    for product in product_catalog:

        score = sum(
            1
            for tag in product["tags"]
            if tag in q
            or any(word in q for word in tag.split())
        )

        if score > 0:
            scored.append((score, product))

    # Highest score first
    scored.sort(key=lambda x: -x[0])

    return [product for _, product in scored[:top_k]]


# ============================================================
# 4. COMMON MODEL FUNCTION
# ============================================================

def ask_model(question, context):

    prompt = f"""
Answer ONLY using the context below.

If the answer is not present in the context,
say "I don't know based on the available information."

Context:
{context}

Question:
{question}
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=150
        )
    )

    return response.text


# ============================================================
# 5. TOOL - ORDER STATUS
# ============================================================

def get_order_status(order_id: str) -> str:
    """
    Look up the live shipping status of an order.

    Args:
        order_id: The customer's order ID, e.g. 8821.

    Returns:
        The current order status.
    """

    fake_orders = {
        "8821": "Out for delivery, expected today by 8pm"
    }

    return fake_orders.get(
        order_id,
        "No order found with that ID"
    )


# ============================================================
# 6. TOOL CALLING
# ============================================================

def ask_model_with_tools(question):

    response = client.models.generate_content(
        model=MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            tools=[get_order_status],
            max_output_tokens=200
        )
    )

    return response.text


# ============================================================
# 7. ROUTER
# ============================================================

def answer(question):

    q = question.lower()

    # --------------------------------------------------------
    # TOOL CALLING
    # --------------------------------------------------------

    if any(
        word in q
        for word in [
            "where is",
            "order #",
            "order status",
            "track"
        ]
    ):
        return ask_model_with_tools(question)

    # --------------------------------------------------------
    # RAG
    # --------------------------------------------------------

    if any(
        word in q
        for word in [
            "suit",
            "recommend",
            "which product"
        ]
    ):

        relevant = retrieve_relevant_products(question)

        if not relevant:
            return "I don't know based on the available product information."

        context = "\n".join(
            f"- {product['name']} "
            f"({', '.join(product['tags'])})"
            for product in relevant
        )

        return ask_model(question, context)

    # --------------------------------------------------------
    # STUFFING
    # --------------------------------------------------------

    return ask_model(question, return_policy)


# ============================================================
# 8. TEST QUESTIONS
# ============================================================

questions = [
    "What's your return policy?",
    "Where is my order #8821?",
    "Which product suits a small kitchen?"
]

for q in questions:

    print("=" * 60)
    print(f"Q: {q}")
    print(f"A: {answer(q)}")

Q: What's your return policy?
A: Returns are accepted within 30 days of delivery. Opened electronics are subject to a 15% restocking fee. Refunds are issued to the original payment method within 5-7 business days.
Q: Where is my order #8821?
A: Your order #8821 is currently out for delivery and is expected to arrive today by 8:00 PM.
Q: Which product suits a small kitchen?
A: Compact 2-Burner Induction Stove and Slim Under-Counter Fridge.
